# ITSN1 Gene Variants and their Association with Parkinson's Disease
## Information
- Project: ITSN1 Burden Analysis
- Date Created: 7/10/25
- Date Last Modified: 3/11/26
- Data: GP2 Release 11 WGS

## Summary

This project explores the *ITSN1* gene variants in the GP2 data.

## Imports

Re-mount resources if not visible in jupyter.

In [1]:
# ! wb resource unmount
# ! wb resource mount

Import the necessary packages 

In [2]:
# System
import os
import sys

# Dataframe and array manipulation
import pandas as pd
import openpyxl
import numpy as np
import math

# Path management
import pathlib
from pathlib import Path
import glob

# Statistics
import statsmodels.api as sm
import scipy
from scipy import stats

# Process management
import shutil
import subprocess

# Workflow management
import papermill as pm

# Visualization
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Date and time
from datetime import datetime

# Get the timestamp
d = datetime.now().strftime("%m%d%Y")
t = datetime.now().strftime("%m%d%Y_%H%M%S")
print("Last iteration date:", d, "EST")
print("Last iteration time:", t, "EST")

# Get package versions
packages = [pd, np, matplotlib, sns, sm, scipy, pm]

print("\nPackage versions:\n")
for p in packages:
    print("-", p.__name__, p.__version__)

Last iteration date: 12102025 EST
Last iteration time: 12102025_231037 EST

Package versions:

- pandas 2.3.2
- numpy 2.2.6
- matplotlib 3.10.1
- seaborn 0.13.2
- statsmodels.api 0.14.4
- scipy 1.15.2
- papermill 2.6.0


### Assign variables for papermill

<div class="alert alert-block alert-info">
<b>Tip:</b> Read papermill documentation here
</div>

[https://papermill.readthedocs.io/en/latest/index.html](https://papermill.readthedocs.io/en/latest/index.html)


<div class="alert alert-block alert-warning">
<b>Example usage:</b> Use the following arguments in an external notebook or in terminal
</div>

```bash
for ANC in EUR AFR EAS AMR AAC AJ CAS MDE SAS FIN; do
  papermill input_notebook.ipynb /path/to/executed/notebooks/output_${ANC}_chr21_GP2_R{RELEASE}_WGS.ipynb \
    -p ANC $ANC \
    -p CHROM chr21 \
    -p GENE ITSN1
done


#### Parameters

<div class="alert alert-block alert-info">
<b>Tip:</b> This notebook uses papermill to externally assign variables.
</div>

<div class="alert alert-block alert-info">
<b>Tip:</b> Change cell tag to 'parameters' so papermill will recognize them.
</div>

In [4]:
# Parameters
ANC = "EUR"
GENE = "ITSN1"
CHROM = "21"
STARTBP = "33542501"
ENDBP = "33999861"
RELEASE = "11"
DT = "wgs"


In [5]:
# Paths

# Home directory
HOME = "/home/jupyter"

# Directory for base results
BASE_RESULTS_DIR = f"{HOME}/{GENE}_R{RELEASE}_Results_{d}"

# Data output directory (for WGS and imputed data)
DT_DIR = f"{BASE_RESULTS_DIR}/{GENE}_{DT.upper()}"

# Work directory for current ANC
WORK_DIR = f"{DT_DIR}/{ANC}"

# Default VEP directory
VEP_DIR = f"{HOME}/vep_data"
os.makedirs(f"{VEP_DIR}/input",exist_ok=True)
os.makedirs(f"{VEP_DIR}/output",exist_ok=True)
# Default workspace directory
DATA_DIR = f"{HOME}/workspace"

# Create directories
dirsToCreate = [BASE_RESULTS_DIR, DT_DIR, WORK_DIR]
for c in dirsToCreate:
    os.makedirs(c, exist_ok=True)
    

## Package Installs

### PLINK

In [6]:
%%bash

mkdir -p ~/tools
cd ~/tools

if test -e /home/jupyter/tools/plink; then
echo "Plink1.9 is already installed in /home/jupyter/tools/"

else
echo -e "Downloading plink \n    -------"
wget -N http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 
unzip -o plink_linux_x86_64_20190304.zip
echo -e "\n plink downloaded and unzipped in /home/jupyter/tools \n "

fi

Plink1.9 is already installed in /home/jupyter/tools/


In [7]:
%%bash

mkdir -p ~/tools
cd ~/tools

if test -e /home/jupyter/tools/plink2; then
echo "Plink2 is already installed in /home/jupyter/tools/"

else
echo -e "Downloading plink2 \n    -------"
wget -N https://s3.amazonaws.com/plink2-assets/alpha5/plink2_linux_x86_64_20240820.zip
unzip -o plink2_linux_x86_64_20240820.zip
echo -e "\n plink2 downloaded and unzipped in /home/jupyter/tools \n "

fi

Plink2 is already installed in /home/jupyter/tools/


### RVTests

In [9]:
%%bash

#Install RVTESTS: Option 1 (~15min)
if test -e /home/jupyter/tools/rvtests; then

echo "rvtests is already installed"
else
echo "rvtests is not installed"

mkdir /home/jupyter/tools/rvtests
cd /home/jupyter/tools/rvtests

wget https://github.com/zhanxw/rvtests/releases/download/v2.1.0/rvtests_linux64.tar.gz 

tar -zxvf rvtests_linux64.tar.gz
fi

rvtests is already installed


In [10]:
# chmod to make sure you have permission to run the program
! chmod u+x /home/jupyter/tools/plink
! chmod u+x /home/jupyter/tools/plink2
! chmod 777 /home/jupyter/tools/rvtests/executable/rvtest

## Covariate File

In [ ]:
# Define columns to keep
key_cols = ['GP2ID', 'wgs_prune_reason', 'diagnosis','baseline_GP2_phenotype', 'baseline_GP2_phenotype_for_qc', 
            'biological_sex_for_qc', 'age_at_sample_collection', 'age_of_onset',  "age_at_diagnosis",'race_for_qc',
            'nba', 'wgs', 'wgs_label', 'study_type']
# Load master key
key = pd.read_csv(f'{DATA_DIR}/gp2_tier2_eu_release{RELEASE}/clinical_data/master_key_release{RELEASE}_final_vwb.csv', 
                  header=0, sep=',', usecols=key_cols).rename(columns = {
                 'GP2ID': 'IID',
                 'baseline_GP2_phenotype':'phenotype',
                 'biological_sex_for_qc':'SEX', 
                 'age_at_sample_collection':'AGE', 
                 'race_for_qc':'RACE',
                 'age_of_onset':'AAO',
                 'age_at_diagnosis':'AAD'})

# Get total count of samples in release
print('Total samples: ', key.shape[0])
# Show heading of the key
key.head()

In [ ]:
## Subset to keep ANC of interest and only wgs samples
ancestry_key = key[(key['wgs_label']==ANC) & (key['wgs']==1)].copy()
print(f"Total in {ANC}: ", ancestry_key.shape[0])
ancestry_key.reset_index(drop=True)

In [14]:
# Check counts per each 
columns = ['study_type', 'diagnosis','phenotype', 'baseline_GP2_phenotype_for_qc']

for c in columns:
    df_counts = ancestry_key[c].value_counts(dropna=False).to_frame(name='counts')
    df_counts.index.name = c
    print(df_counts, '\n')

                      counts
study_type                  
Case(/Control)         13616
Brain Bank              3458
Monogenic               2228
Prodromal                822
Genetically Enriched     720 

                                                    counts
diagnosis                                                 
PD                                                    9859
Parkinson's Disease                                   2728
No PD Nor Other Neurological Disorder                 1324
Control                                               1173
Prodromal                                              775
...                                                    ...
Parkinson's disease;  Alzheimer's disease                1
Parkinson's disease; argyrophilic grains, parah...       1
Parkinson disease; Dementia (history); acute in...       1
Parkinson disease;  Alzheimer's disease; Hippoc...       1
Control;  Alzheimer type II astrocytosis, consi...       1

[568 rows x 1 columns] 

  

In [ ]:
keep_studytype = ['Monogenic', 'Case(/Control)', 'Brain Bank', 'Genetically Enriched']
keep_phenotype = ['pd', 'control']

In [ ]:
# filter to include wanted study types
print("Before filter: ", ancestry_key.shape[0])
ancestry_key = ancestry_key[
    (ancestry_key["study_type"].isin(keep_studytype)) &
    (ancestry_key["phenotype"].str.contains('|'.join(keep_phenotype), case=False))
]
print("After filter: ", ancestry_key.shape[0])

Before filter:  20844


After filter:  16625


In [17]:
for c in columns:
    df_counts = ancestry_key[c].value_counts(dropna=False).to_frame(name='counts')
    df_counts.index.name = c
    print(df_counts, '\n')

                      counts
study_type                  
Case(/Control)         12656
Monogenic               1874
Brain Bank              1696
Genetically Enriched     399 

                                       counts
diagnosis                                    
PD                                       9850
Parkinson's Disease                      2728
Control                                  1125
No PD Nor Other Neurological Disorder    1114
Parkinson's disease                       458
...                                       ...
PD unaffected                               1
PRKN-PD                                     1
PD + DRD                                    1
PD + ET + possible dementia                 1
LRRK2-PD                                    1

[201 rows x 1 columns] 

             counts
phenotype          
PD            11568
Control        2784
Affected_PD    2273 

                               counts
baseline_GP2_phenotype_for_qc        
PD                    

In [18]:
# Convert phenotype to binary (1/2) and drop NAs

pheno_mapping = {"PD": 2, "Affected_PD":2, "Control": 1}
ancestry_key['PHENO'] = ancestry_key['phenotype'].map(pheno_mapping).astype('Int64')

# drop NAs
ancestry_key = ancestry_key[ancestry_key['PHENO'].notna()]

# Check value counts of pheno
ancestry_key['PHENO'].value_counts(dropna=False)

PHENO
2    13841
1     2784
Name: count, dtype: Int64

In [19]:
# Load information about related individuals 
try:
    print("Before: ", ancestry_key.shape[0])
    related_df = pd.read_csv(f'{DATA_DIR}/gp2_tier2_eu_release{RELEASE}/{DT.lower()}/deepvariant_joint_calling/related_samples/{ANC}/{ANC}_release{RELEASE}.related')
    print("Related detected: ", related_df.shape[0])
    # Make a list of just one set of related people
    related_list = list(related_df['IID1'])

    # remove related individuals
    print(f"Removing {len(related_list)} individuals.")

    ancestry_key = ancestry_key[~ancestry_key["IID"].isin(related_list)]

    # Check size
    print("Unrelated dataset: ", ancestry_key.shape[0])
except:
    print("Warning. No related samples file found. Removing 0 samples.")


Before:  16625


Related detected:  1775
Removing 1775 individuals.


Unrelated dataset:  16001


In [21]:
# Assign conditions so female=2 and men=1, and -9 otherwise (matching PLINK convention)
# Female = 2; Male = 1
sex_mapping = {"Female": 2, "Male": 1}
ancestry_key["SEX"] = ancestry_key['SEX'].map(sex_mapping).astype('Int64')

# Check value counts of SEX
ancestry_key["SEX"].value_counts(dropna=False)

SEX
1       9645
2       6349
<NA>       7
Name: count, dtype: Int64

In [22]:
# check that we are only including samples with available wgs

print(f"Retaining {(ancestry_key['wgs'] == 1).sum()} samples with {DT.lower()} and {(ancestry_key['wgs'] != 1).sum()} without {DT.lower()}")

ancestry_key = ancestry_key[ancestry_key["wgs"] == 1]

Retaining 16001 samples with wgs and 0 without wgs


In [ ]:
## Get the PCs
pcs = pd.read_csv(f'{DATA_DIR}/gp2_tier2_eu_release{RELEASE}/{DT.lower()}/deepvariant_joint_calling/pcs/{ANC}/{ANC}_release{RELEASE}.eigenvec', sep='\t')
selected_columns = ['IID', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
pcs = pd.DataFrame(data=pcs.iloc[:, 1:7].values, columns=selected_columns)

# Drop the first row (since it's now the column names)
pcs = pcs.drop(0)

# Reset the index to remove any potential issues
pcs = pcs.reset_index(drop=True)

# Display the resulting DataFrame
print(pcs)

In [24]:
## Check for samples missing pcs and pheno
df = pd.merge(pcs, ancestry_key, on='IID', how="inner")

print(f"Dropping {ancestry_key.shape[0] - df.shape[0]} samples with no pcs")


## Drop lines with missing pheno
print(f"Dropping {(df['PHENO'].isna()).sum()} samples with missing pheno")

df = df[df['PHENO'].notna()]

df["PHENO"].value_counts(dropna=False)


Dropping 1 samples with no pcs
Dropping 0 samples with missing pheno


PHENO
2    13413
1     2587
Name: count, dtype: Int64

In [25]:
# get FID from .psam file 
FID_df = pd.read_csv(f"{DATA_DIR}/gp2_tier2_eu_release{RELEASE}/{DT.lower()}/deepvariant_joint_calling/plink/{ANC}/chr{CHROM}_{ANC}_release{RELEASE}.psam", sep = "\t", usecols=["#FID", "IID"])
df = df.merge(FID_df, how="left", on="IID")

In [26]:
## Clean up and keep columns we need 
final_df = df[['#FID','IID', 'SEX', 'AGE', 'PHENO', 'AAO', 'AAD', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']].copy()
final_df.groupby(['PHENO'])['SEX'].value_counts(dropna=False)

PHENO  SEX 
1      2       1353
       1       1234
2      1       8411
       2       4995
       <NA>       7
Name: count, dtype: Int64

In [27]:
## Make file of sample IDs to keep 
samples_toKeep = final_df[['#FID', 'IID']].copy()

samples_toKeep.to_csv(f'{WORK_DIR}/{ANC}.samplestoKeep', sep = '\t', index=False, header=None)
samples_toKeep.shape

(16000, 2)

In [28]:
## Save your covariate file
final_df.to_csv(f'{WORK_DIR}/{ANC}_covariate_file.txt', sep = '\t', index=False, na_rep= "NA", header=True)

### Get cohort summary stats

In [30]:
print(f"""
Number of {ANC} PD cases: {(final_df["PHENO"] == 2).sum()}
Number of {ANC} Controls: {(final_df["PHENO"] == 1).sum()}

PD age range: {round(final_df[final_df["PHENO"] == 2]["AGE"].mean(),2)} +/- {round(final_df[final_df["PHENO"] == 2]["AGE"].std(),2)}
Controls age range: {round(final_df[final_df["PHENO"] == 1]["AGE"].mean(),2)} +/- {round(final_df[final_df["PHENO"] == 1]["AGE"].std(),2)}

Cases in AAO analysis: {((final_df["PHENO"] == 2) & (final_df["AAO"].notna())).sum()}

nan_counts:\n{final_df.isna().sum()}
""")


Number of EUR PD cases: 13413
Number of EUR Controls: 2587

PD age range: 63.47 +/- 11.59
Controls age range: 65.81 +/- 12.39

Cases in AAO analysis: 9762

nan_counts:
#FID        0
IID         0
SEX         7
AGE      2469
PHENO       0
AAO      6231
AAD      6542
PC1         0
PC2         0
PC3         0
PC4         0
PC5         0
dtype: int64



## Subset PLINK ANC file

In [31]:
PLINK_PREFIX = f"chr{CHROM}_{GENE}"
print(PLINK_PREFIX)

chr21_ITSN1


In [ ]:
## extract region of interest from file, flanked by 100kb both sides
! /home/jupyter/tools/plink2 \
--pfile {DATA_DIR}/gp2_tier2_eu_release{RELEASE}/{DT.lower()}/deepvariant_joint_calling/plink/{ANC}/chr{CHROM}_{ANC}_release{RELEASE} \
--chr {CHROM} \
--from-bp {STARTBP} \
--to-bp {ENDBP} \
--make-pgen erase-dosage \
--out {WORK_DIR}/{PLINK_PREFIX}


In [33]:
# filter variants to just including those with mac1, and by samples to Keep to make main file for analyses
! /home/jupyter/tools/plink2 \
--pfile {WORK_DIR}/{PLINK_PREFIX} \
--keep {WORK_DIR}/{ANC}.samplestoKeep \
--mac 1 \
--make-pgen \
--out {WORK_DIR}/chr{CHROM}_{ANC}

PLINK v2.0.0-a.6.24LM 64-bit Intel (15 Sep 2025)   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/chr21_EUR.log.
Options in effect:
  --keep /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR.samplestoKeep
  --mac 1
  --make-pgen
  --out /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/chr21_EUR
  --pfile /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/chr21_ITSN1

Start time: Wed Dec 10 23:10:50 2025
12962 MiB RAM detected, ~10780 available; reserving 6481 MiB for main
workspace.
Using up to 2 compute threads.
20844 samples (8572 females, 12242 males, 30 ambiguous; 20844 founders) loaded
from /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/chr21_ITSN1.psam.
37485 variants loaded from
/home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/chr21_ITSN1.pvar.
Note: No phenotype data present.
--keep: 16000 samples remaining.
16000 samples

done.
End time: Wed Dec 10 23:10:50 2025


In [ ]:
## generate individual pfile for variant annotation

# Get one sample 
samples = pd.read_csv(f"{WORK_DIR}/{ANC}.samplestoKeep", sep = "\t", header=None)
sample_id_1 = samples.iloc[0,0]
sample_id_2 = samples.iloc[0,1]
sample_id = f"{sample_id_1} {sample_id_2}"


# convert to vcf for annotation
! /home/jupyter/tools/plink2 \
--pfile {WORK_DIR}/chr{CHROM}_{ANC} \
--recode vcf id-paste=iid \
--indv {sample_id} \
--out {WORK_DIR}/{ANC}_vep_input

In [35]:
# move to input directory
os.makedirs(f"{VEP_DIR}/input/R{RELEASE}_{DT}", exist_ok=True)
! cp {WORK_DIR}/{ANC}_vep_input.vcf {VEP_DIR}/input/R{RELEASE}_{DT}/{ANC}_vep_input.vcf

In [36]:
### Bgzip and Tabix (zip and index the file)
! bgzip -k -f {WORK_DIR}/{ANC}_vep_input.vcf
! tabix -f -p vcf {WORK_DIR}/{ANC}_vep_input.vcf.gz
! tabix -f -p vcf {WORK_DIR}/{ANC}_vep_input.vcf.gz

## Annotation using VEP
- *ITSN1* from NCBI gene, with gene region flanked by 200kb on both sides
- hg38 (chr21:33442501-34099861) 

In [37]:
# run container to generate VEP output
vep_cmd = [
    "vep",
    "--cache",
    "--offline",
    "--force_overwrite",
    "--input_file", f"{VEP_DIR}/input/R{RELEASE}_{DT}/{ANC}_vep_input.vcf",
    "--output_file", f"{VEP_DIR}/output/{ANC}_R{RELEASE}_{DT.lower()}_vep_output.txt",
    "--refseq",
    "--flag_pick_allele",
    "--no_stats",
    "--hgvs", # get hgvs annotations 
    "--use_transcript_ref",
    "--assembly", "GRCh38",
    "--fasta", f"{HOME}/.vep/homo_sapiens/115_GRCh38/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz",
    "--force_overwrite"
]

try:
    subprocess.run(vep_cmd, check=True)
except Exception as e:
    print(f"Error: {e}")

2025-12-10 23:10:57 - INFO: BAM-edited cache detected, enabling --use_transcript_ref; use --use_given_ref to override this


In [38]:
# remove header lines
! grep -v "##" {VEP_DIR}/output/{ANC}_R{RELEASE}_{DT.lower()}_vep_output.txt > {WORK_DIR}/{ANC}_R{RELEASE}_{DT.lower()}_vep_output.txt

In [39]:
# filter to variants flagged by pick
! awk 'NR==1 || /PICK/' {WORK_DIR}/{ANC}_R{RELEASE}_{DT.lower()}_vep_output.txt > {WORK_DIR}/{ANC}_R{RELEASE}_{DT.lower()}_vep_output_filtered_pick.txt

In [40]:
# Load VEP output in pd df
# Skip commented lines
# Select columns of interest
gene = pd.read_csv(f"{WORK_DIR}/{ANC}_R{RELEASE}_{DT.lower()}_vep_output_filtered_pick.txt", sep='\t', engine='pyarrow')
gene.head()

,#Uploaded_variation,Location,Allele,Gene,Feature,Feature_type,Consequence,cDNA_position,CDS_position,Protein_position,Amino_acids,Codons,Existing_variation,Extra
0,chr21:33542504:G:C,21:33542504,C,6651,NM_138927.4,Transcript,upstream_gene_variant,-,-,-,-,-,-,IMPACT=MODIFIER;DISTANCE=534;STRAND=1;PICK=1;G...
1,chr21:33542512:G:A,21:33542512,A,6651,NM_138927.4,Transcript,upstream_gene_variant,-,-,-,-,-,-,IMPACT=MODIFIER;DISTANCE=526;STRAND=1;PICK=1;G...
2,chr21:33542514:T:TA,21:33542514-33542515,A,6651,NM_138927.4,Transcript,upstream_gene_variant,-,-,-,-,-,-,IMPACT=MODIFIER;DISTANCE=523;STRAND=1;PICK=1;G...
3,chr21:33542514:TA:T,21:33542515,-,6651,NM_138927.4,Transcript,upstream_gene_variant,-,-,-,-,-,-,IMPACT=MODIFIER;DISTANCE=523;STRAND=1;PICK=1;G...
4,chr21:33542515:A:G,21:33542515,G,6651,NM_138927.4,Transcript,upstream_gene_variant,-,-,-,-,-,-,IMPACT=MODIFIER;DISTANCE=523;STRAND=1;PICK=1;G...


In [41]:
# ref seq ID for ITSN1 is 6453, filter for it to keep only ITSN1 variants
gene_subset = gene[gene["Gene"] == "6453"].copy()
print("ITSN1 variants: ", gene_subset.shape[0])

ITSN1 variants:  18832


In [42]:
# split extra information into its own dataframe
extra_df = gene_subset["Extra"].str.split(";", expand=True).copy()
extra_df = extra_df.rename(columns={0:"Impact", 1:"Distance", 2:"Strand", 3:"PICK", 4:"Given_Ref", 5:"Used_Ref"})
extra_df

,Impact,Distance,Strand,PICK,Given_Ref,Used_Ref,6,7
7147,IMPACT=MODIFIER,DISTANCE=756,STRAND=1,PICK=1,GIVEN_REF=G,USED_REF=G,None,None
7148,IMPACT=MODIFIER,DISTANCE=752,STRAND=1,PICK=1,GIVEN_REF=C,USED_REF=C,None,None
7149,IMPACT=MODIFIER,DISTANCE=740,STRAND=1,PICK=1,GIVEN_REF=C,USED_REF=C,None,None
7150,IMPACT=MODIFIER,DISTANCE=729,STRAND=1,PICK=1,GIVEN_REF=G,USED_REF=G,None,None
7151,IMPACT=MODIFIER,DISTANCE=714,STRAND=1,PICK=1,GIVEN_REF=C,USED_REF=C,None,None
...,...,...,...,...,...,...,...,...
25974,IMPACT=MODIFIER,DISTANCE=3533,STRAND=1,PICK=1,GIVEN_REF=A,USED_REF=A,None,None
25975,IMPACT=MODIFIER,DISTANCE=3556,STRAND=1,PICK=1,GIVEN_REF=T,USED_REF=T,None,None
25976,IMPACT=MODIFIER,DISTANCE=3557,STRAND=1,PICK=1,GIVEN_REF=T,USED_REF=T,None,None
25977,IMPACT=MODIFIER,DISTANCE=3583,STRAND=1,PICK=1,GIVEN_REF=A,USED_REF=A,None,None


In [43]:
# get hgvs annotations
hgvs_df = extra_df[6].str.split(":", expand=True).copy()
hgvs_df = hgvs_df.rename(columns={1:"HGVS"})
hgvs_df

,0,HGVS
7147,None,None
7148,None,None
7149,None,None
7150,None,None
7151,None,None
...,...,...
25974,None,None
25975,None,None
25976,None,None
25977,None,None


In [44]:
# drop unnecessary columns
gene_subset = gene_subset.drop(columns=["Extra", "cDNA_position","CDS_position", "Protein_position", "Amino_acids", "Codons", "Existing_variation"])
gene_subset

,#Uploaded_variation,Location,Allele,Gene,Feature,Feature_type,Consequence
7147,chr21:33641745:G:A,21:33641745,A,6453,NM_003024.3,Transcript,upstream_gene_variant
7148,chr21:33641749:C:T,21:33641749,T,6453,NM_003024.3,Transcript,upstream_gene_variant
7149,chr21:33641761:C:T,21:33641761,T,6453,NM_003024.3,Transcript,upstream_gene_variant
7150,chr21:33641772:G:A,21:33641772,A,6453,NM_003024.3,Transcript,upstream_gene_variant
7151,chr21:33641787:C:T,21:33641787,T,6453,NM_003024.3,Transcript,upstream_gene_variant
...,...,...,...,...,...,...,...
25974,chr21:33903394:A:G,21:33903394,G,6453,NM_003024.3,Transcript,downstream_gene_variant
25975,chr21:33903417:T:G,21:33903417,G,6453,NM_003024.3,Transcript,downstream_gene_variant
25976,chr21:33903418:T:C,21:33903418,C,6453,NM_003024.3,Transcript,downstream_gene_variant
25977,chr21:33903444:A:C,21:33903444,C,6453,NM_003024.3,Transcript,downstream_gene_variant


In [ ]:
# save df of extra information for appending later
final_extra_df = pd.concat([gene_subset, extra_df.drop(columns=[6]), hgvs_df.drop(columns=0)], axis=1).rename(columns={"#Uploaded_variation":"SNP"})
final_extra_df.to_csv(f"{WORK_DIR}/vep_annotations_filtered_pick_extra_columns.txt", sep="\t", index=False,)

In [46]:
final_extra_df

,SNP,Location,Allele,Gene,Feature,Feature_type,Consequence,Impact,Distance,Strand,PICK,Given_Ref,Used_Ref,7,HGVS
7147,chr21:33641745:G:A,21:33641745,A,6453,NM_003024.3,Transcript,upstream_gene_variant,IMPACT=MODIFIER,DISTANCE=756,STRAND=1,PICK=1,GIVEN_REF=G,USED_REF=G,None,None
7148,chr21:33641749:C:T,21:33641749,T,6453,NM_003024.3,Transcript,upstream_gene_variant,IMPACT=MODIFIER,DISTANCE=752,STRAND=1,PICK=1,GIVEN_REF=C,USED_REF=C,None,None
7149,chr21:33641761:C:T,21:33641761,T,6453,NM_003024.3,Transcript,upstream_gene_variant,IMPACT=MODIFIER,DISTANCE=740,STRAND=1,PICK=1,GIVEN_REF=C,USED_REF=C,None,None
7150,chr21:33641772:G:A,21:33641772,A,6453,NM_003024.3,Transcript,upstream_gene_variant,IMPACT=MODIFIER,DISTANCE=729,STRAND=1,PICK=1,GIVEN_REF=G,USED_REF=G,None,None
7151,chr21:33641787:C:T,21:33641787,T,6453,NM_003024.3,Transcript,upstream_gene_variant,IMPACT=MODIFIER,DISTANCE=714,STRAND=1,PICK=1,GIVEN_REF=C,USED_REF=C,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25974,chr21:33903394:A:G,21:33903394,G,6453,NM_003024.3,Transcript,downstream_gene_variant,IMPACT=MODIFIER,DISTANCE=3533,STRAND=1,PICK=1,GIVEN_REF=A,USED_REF=A,None,None
25975,chr21:33903417:T:G,21:33903417,G,6453,NM_003024.3,Transcript,downstream_gene_variant,IMPACT=MODIFIER,DISTANCE=3556,STRAND=1,PICK=1,GIVEN_REF=T,USED_REF=T,None,None
25976,chr21:33903418:T:C,21:33903418,C,6453,NM_003024.3,Transcript,downstream_gene_variant,IMPACT=MODIFIER,DISTANCE=3557,STRAND=1,PICK=1,GIVEN_REF=T,USED_REF=T,None,None
25977,chr21:33903444:A:C,21:33903444,C,6453,NM_003024.3,Transcript,downstream_gene_variant,IMPACT=MODIFIER,DISTANCE=3583,STRAND=1,PICK=1,GIVEN_REF=A,USED_REF=A,None,None


In [47]:
# edit dataframe to create variants to keep file that matches plink format
gene_subset["Location"] = gene_subset["Location"].str.replace("21:", "")
variation_df = gene_subset["#Uploaded_variation"].str.split(":", expand=True)
location_df = gene_subset["Location"].str.split("-", expand=True)

location_df.columns = ["Start", "End"]
location_df["End"] = location_df.apply(lambda row: row["Start"] if row["End"] is None else row["End"], axis=1)

variation_df.columns = ["Chr", "Start", "Ref", "Alt"]
variation_df.drop(columns=["Start"], inplace=True)

fixed_columns = pd.concat([variation_df, location_df], axis=1)
gene_df = pd.concat([fixed_columns,gene_subset], axis=1)

# drop unnecessary columns
gene_df.drop(columns=["Location", "Feature", "Feature_type"], axis=1, inplace=True)

gene_df["Gene"] = "ITSN1"
gene_df.rename(columns={"#Uploaded_variation":"SNP"}, inplace=True)

In [48]:
# get value counts to determine how many total annotated
variant_counts = gene_df["Consequence"].value_counts().rename_axis('Consequence').reset_index(name='counts').assign(
    Consequence=lambda x: x['Consequence'].str.replace('_', ' '))
variant_counts

,Consequence,counts
0,intron variant,17276
1,3 prime UTR variant,808
2,downstream gene variant,275
3,missense variant,150
4,synonymous variant,108
5,upstream gene variant,104
6,5 prime UTR variant,30
7,"splice polypyrimidine tract variant,intron var...",22
8,"splice region variant,splice polypyrimidine tr...",20
9,"missense variant,splice region variant",8


In [49]:
# write value_counts output to file
variant_counts.to_csv(f"{WORK_DIR}/{ANC}_{GENE}_all_vep_consequences.txt", sep="\t", index=False)

In [50]:
# filter consequences to only missense, splicing, frameshift and stopgain, and other high priority consequences
# Source: https://useast.ensembl.org/info/genome/variation/prediction/predicted_data.html 

vep_coding_variants = gene_df[gene_df["Consequence"].str.contains('^splice_donor_variant|^splice_acceptor_variant|^missense|^stop_gained|frame|^protein_altering|transcript_ablation|transcript_amplification', regex=True)].copy()
vep_coding_variants["Consequence"].value_counts()

Consequence
missense_variant                          150
missense_variant,splice_region_variant      8
frameshift_variant                          6
stop_gained                                 5
splice_donor_variant                        4
inframe_deletion                            2
Name: count, dtype: int64

In [51]:
# filter out any synonymous variants
final_vep_coding_variants = vep_coding_variants[~(vep_coding_variants["Consequence"].str.contains('synonymous_variant', regex=False))].copy()
final_vep_coding_variants["Consequence"].value_counts()

Consequence
missense_variant                          150
missense_variant,splice_region_variant      8
frameshift_variant                          6
stop_gained                                 5
splice_donor_variant                        4
inframe_deletion                            2
Name: count, dtype: int64

In [52]:
final_vep_coding_variants

,Chr,Ref,Alt,Start,End,SNP,Allele,Gene,Consequence
13007,chr21,C,T,33721217,33721217,chr21:33721217:C:T,T,ITSN1,missense_variant
13009,chr21,A,G,33721223,33721223,chr21:33721223:A:G,G,ITSN1,missense_variant
13108,chr21,CT,C,33722605,33722605,chr21:33722604:CT:C,-,ITSN1,frameshift_variant
13986,chr21,G,A,33735066,33735066,chr21:33735066:G:A,A,ITSN1,missense_variant
13987,chr21,G,A,33735084,33735084,chr21:33735084:G:A,A,ITSN1,missense_variant
...,...,...,...,...,...,...,...,...,...
24886,chr21,C,T,33888182,33888182,chr21:33888182:C:T,T,ITSN1,missense_variant
24888,chr21,C,T,33888221,33888221,chr21:33888221:C:T,T,ITSN1,missense_variant
24891,chr21,T,C,33888260,33888260,chr21:33888260:T:C,C,ITSN1,missense_variant
24893,chr21,C,A,33888296,33888296,chr21:33888296:C:A,A,ITSN1,missense_variant


In [53]:
# Save ids in PLINK format
final_vep_coding_variants["SNP"].to_csv(f"{WORK_DIR}/{ANC}_{GENE}.all_variants_toKeep.txt", sep="\t",  index=False, header=False)

In [54]:
# save consequences annotation file for later
final_vep_coding_variants.to_csv(f"{WORK_DIR}/{ANC}_{GENE}_vep_coding_variant_annotations.txt",  sep="\t",  index=False)

## Burden Analyses using RVTests


In [55]:
PFILE_PREFIX = f"chr{CHROM}_{ANC}"

In [56]:
# get hg38 refFlat file from ucsc
! wget -nc -O /home/jupyter/refFlat.txt.gz https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/refFlat.txt.gz
! gunzip -q -f -k /home/jupyter/refFlat.txt.gz

File ‘/home/jupyter/refFlat.txt.gz’ already there; not retrieving.


In [57]:
# make a pheno file for plink input
! cut -f 1,2,5 {WORK_DIR}/{ANC}_covariate_file.txt > {WORK_DIR}/{ANC}_pheno.txt

In [58]:
# Convert the files from Plink 2.0 to Plink 1.9 format 
! /home/jupyter/tools/plink2 \
--pfile {WORK_DIR}/{PFILE_PREFIX} \
--make-bed \
--pheno {WORK_DIR}/{ANC}_pheno.txt \
--pheno-name PHENO \
--max-alleles 2 \
--out {WORK_DIR}/{PFILE_PREFIX}

PLINK v2.0.0-a.6.24LM 64-bit Intel (15 Sep 2025)   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/chr21_EUR.log.
Options in effect:
  --make-bed
  --max-alleles 2
  --out /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/chr21_EUR
  --pfile /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/chr21_EUR
  --pheno /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_pheno.txt
  --pheno-name PHENO

Start time: Wed Dec 10 23:16:39 2025
12962 MiB RAM detected, ~10735 available; reserving 6481 MiB for main
workspace.
Using up to 2 compute threads.
16000 samples (6350 females, 9643 males, 7 ambiguous; 16000 founders) loaded
from /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/chr21_EUR.psam.
32814 variants loaded from
/home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/chr21_EUR.pvar.
1 binary phenotype loaded (13413 cases, 2587 controls).
32814 

done.
End time: Wed Dec 10 23:16:40 2025


In [59]:
## extract variants as a vcf
! /home/jupyter/tools/plink \
--bfile {WORK_DIR}/{PFILE_PREFIX} \
--keep {WORK_DIR}/{ANC}.samplestoKeep \
--extract {WORK_DIR}/{ANC}_{GENE}.all_variants_toKeep.txt \
--recode vcf-iid \
--out {WORK_DIR}/{ANC}_{GENE}.coding_nonsyn

PLINK v1.9.0-b.7.11 64-bit (19 Aug 2025)           cog-genomics.org/plink/1.9/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1.coding_nonsyn.log.
Options in effect:
  --bfile /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/chr21_EUR
  --extract /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1.all_variants_toKeep.txt
  --keep /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR.samplestoKeep
  --out /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1.coding_nonsyn
  --recode vcf-iid

12962 MB RAM detected; reserving 6481 MB for main workspace.
32814 variants loaded from .bim file.
16000 people (9643 males, 6350 females, 7 ambiguous) loaded from .fam.
Ambiguous sex IDs written to
/home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1.coding_nonsyn.nosex
.
16000 phenotype values loaded from .fam.
--extract: 175 variants remaining.
-

74%75%76%77%78%79%80%81%82%83%84%85%86%87%88%89%90%91%92%93%94%95%96%97%98%99%done.


In [60]:
### Bgzip and Tabix (zip and index the file)
! bgzip -f {WORK_DIR}/{ANC}_{GENE}.coding_nonsyn.vcf
! tabix -f -p vcf {WORK_DIR}/{ANC}_{GENE}.coding_nonsyn.vcf.gz

In [61]:
# because of errors with rvtests reading in phenotypes, create cov file that matches rvtest pheno file header
rv_df = pd.read_csv(f'{WORK_DIR}/{ANC}_covariate_file.txt', sep="\t")
rv_df.columns = rv_df.columns.str.lower()
rv_df["fid"] = rv_df["#fid"]
rv_df["fatid"] = 0
rv_df["matid"] = 0
rv_df = rv_df[["fid", "iid", "fatid", "matid", "sex", "pheno", "age", "pc1", "pc2", "pc3", "pc4", "pc5"]]
rv_df.to_csv(f'{WORK_DIR}/rvtests_covariate_file.txt', sep='\t', index=False, header=True)

In [ ]:
## RVtests with covariates 
! /home/jupyter/tools/rvtests/executable/rvtest --noweb --hide-covar \
--out {WORK_DIR}/{ANC}_{GENE}.burden.coding_nonsyn \
--kernel skat,skato \
--inVcf {WORK_DIR}/{ANC}_{GENE}.coding_nonsyn.vcf.gz \
--pheno {WORK_DIR}/rvtests_covariate_file.txt \
--pheno-name pheno \
--gene {GENE} \
--geneFile /home/jupyter/refFlat.txt \
--covar {WORK_DIR}/rvtests_covariate_file.txt \
--covar-name sex,pc1,pc2,pc3,pc4,pc5

# --out : Name of output 
# --burden cmc --kernel skato: tests to run 
# --inVcf : VCF file 
# --gene: gene name (if only looking at one or a few)
# --geneFile refFlat.txt
# --pheno :  covar file
# --mpheno : # column that has phenotype information
# --pheno-name : column name with phenotype in file
# --covar : covar file
# --freqUpper : optional, MAF cut-off
# --covar-name : covariates, listed by column name, separated by commas (no spaces between commas)
## 1=controls; 2=cases

## Case/Control Frequencies

### Run PLINK2 GLM

In [67]:
# run logistic regression for pvals and odds ratios
! /home/jupyter/tools/plink2 \
--pfile {WORK_DIR}/{PFILE_PREFIX} \
--adjust \
--keep {WORK_DIR}/{ANC}.samplestoKeep \
--pheno {WORK_DIR}/{ANC}_pheno.txt \
--extract {WORK_DIR}/{ANC}_{GENE}.all_variants_toKeep.txt \
--ci 0.95 \
--covar {WORK_DIR}/{ANC}_covariate_file.txt \
--covar-name SEX,PC1,PC2,PC3,PC4,PC5 \
--covar-variance-standardize \
--glm hide-covar omit-ref firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+gcountcc,+err \
--out {WORK_DIR}/{ANC}_{GENE}

PLINK v2.0.0-a.6.24LM 64-bit Intel (15 Sep 2025)   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1.log.
Options in effect:
  --adjust
  --ci 0.95
  --covar /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_covariate_file.txt
  --covar-name SEX,PC1,PC2,PC3,PC4,PC5
  --covar-variance-standardize
  --extract /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1.all_variants_toKeep.txt
  --glm hide-covar omit-ref firth-fallback cols=+a1freq,+a1freqcc,+a1count,+totallele,+a1countcc,+totallelecc,+gcountcc,+err
  --keep /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR.samplestoKeep
  --out /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1
  --pfile /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/chr21_EUR
  --pheno /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_pheno.txt

Start time: Wed Dec 10 23

done.
Results written to /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1.PHENO.glm.logistic.hybrid .
--adjust: Genomic inflation est. lambda (based on median chisq) = 0.477895.
(Treating lambda as 1 in GC-corrected p-value calculation.)
--adjust values (175 tests) written to
/home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1.PHENO.glm.logistic.hybrid.adjusted
.
End time: Wed Dec 10 23:17:03 2025


### Run Basic Association Test (fisher and 1df chi-sq)

In [68]:
BFILE_PREFIX = f"chr{CHROM}_{ANC}"

In [69]:
## Run association test using Fisher's exact test
! /home/jupyter/tools/plink \
--bfile {WORK_DIR}/{BFILE_PREFIX} \
--extract {WORK_DIR}/{ANC}_{GENE}.all_variants_toKeep.txt \
--keep {WORK_DIR}/{ANC}.samplestoKeep \
--pheno {WORK_DIR}/{ANC}_pheno.txt \
--mpheno 1 \
--assoc fisher \
--ci 0.95 \
--allow-no-sex \
--out {WORK_DIR}/{ANC}_{GENE}

PLINK v1.9.0-b.7.11 64-bit (19 Aug 2025)           cog-genomics.org/plink/1.9/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1.log.
Options in effect:
  --allow-no-sex
  --assoc fisher
  --bfile /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/chr21_EUR
  --ci 0.95
  --extract /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1.all_variants_toKeep.txt
  --keep /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR.samplestoKeep
  --mpheno 1
  --out /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1
  --pheno /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_pheno.txt

12962 MB RAM detected; reserving 6481 MB for main workspace.
32814 variants loaded from .bim file.
16000 people (9643 males, 6350 females, 7 ambiguous) loaded from .fam.
Ambiguous sex IDs written to
/home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1.nosex .


### Get variant frequencies

In [70]:
! /home/jupyter/tools/plink \
--bfile {WORK_DIR}/{BFILE_PREFIX} \
--extract {WORK_DIR}/{ANC}_{GENE}.all_variants_toKeep.txt \
--keep {WORK_DIR}/{ANC}.samplestoKeep \
--allow-no-sex \
--freq \
--out {WORK_DIR}/{ANC}_{GENE}

PLINK v1.9.0-b.7.11 64-bit (19 Aug 2025)           cog-genomics.org/plink/1.9/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1.log.
Options in effect:
  --allow-no-sex
  --bfile /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/chr21_EUR
  --extract /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1.all_variants_toKeep.txt
  --freq
  --keep /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR.samplestoKeep
  --out /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1

12962 MB RAM detected; reserving 6481 MB for main workspace.
32814 variants loaded from .bim file.
16000 people (9643 males, 6350 females, 7 ambiguous) loaded from .fam.
Ambiguous sex IDs written to
/home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1.nosex .
16000 phenotype values loaded from .fam.
--extract: 175 variants remaining.
--keep: 16000 people remaining.
Usin

### Merge files together

In [71]:
! /home/jupyter/tools/plink \
--bfile {WORK_DIR}/{BFILE_PREFIX} \
--extract {WORK_DIR}/{ANC}_{GENE}.all_variants_toKeep.txt \
--keep {WORK_DIR}/{ANC}.samplestoKeep \
--pheno {WORK_DIR}/{ANC}_pheno.txt \
--pheno-name PHENO \
--allow-no-sex \
--recode A \
--out {WORK_DIR}/{ANC}_{GENE}

PLINK v1.9.0-b.7.11 64-bit (19 Aug 2025)           cog-genomics.org/plink/1.9/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1.log.
Options in effect:
  --allow-no-sex
  --bfile /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/chr21_EUR
  --extract /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1.all_variants_toKeep.txt
  --keep /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR.samplestoKeep
  --out /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1
  --pheno /home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_pheno.txt
  --pheno-name PHENO
  --recode A

12962 MB RAM detected; reserving 6481 MB for main workspace.
32814 variants loaded from .bim file.
16000 people (9643 males, 6350 females, 7 ambiguous) loaded from .fam.
Ambiguous sex IDs written to
/home/jupyter/ITSN1_R11_Results_12102025/ITSN1_WGS/EUR/EUR_ITSN1.nosex .
16000 ph

41%42%43%44%45%46%47%48%49%50%51%52%53%54%55%56%57%58%59%60%61%62%63%64%65%66%67%68%69%70%71%72%73%74%75%76%77%78%79%80%81%82%83%84%85%86%87%88%89%90%91%92%93%94%95%96%97%98%99%done.


In [72]:
### merge output files

ASSOC_FILE = f"{WORK_DIR}/{ANC}_{GENE}.assoc.fisher"
RECODE_FILE = f"{WORK_DIR}/{ANC}_{GENE}.raw"
GLM_HYBRID_FILE = f"{WORK_DIR}/{ANC}_{GENE}.PHENO.glm.logistic.hybrid"
GLM_ADJUST_FILE = f"{WORK_DIR}/{ANC}_{GENE}.PHENO.glm.logistic.hybrid.adjusted"
FREQ_FILE = f"{WORK_DIR}/{ANC}_{GENE}.frq"


log_hybrid = pd.read_csv(GLM_HYBRID_FILE, sep="\s+", usecols=["ID", "A1", "A1_FREQ", "A1_CT", "ALLELE_CT", "P", "OR", "LOG(OR)_SE", "L95", "U95", "FIRTH?", "ERRCODE"])
log_hybrid.rename(columns={"ID":"SNP"}, inplace=True)

assoc_adjusted = pd.read_csv(GLM_ADJUST_FILE,  sep="\s+", usecols=["ID", "BONF"])
assoc_adjusted.rename(columns={"ID":"SNP"}, inplace=True)

assoc = pd.read_csv(ASSOC_FILE, sep="\s+", usecols=["SNP", "P", "F_A", "F_U"])
assoc.rename(columns={"P": "P_fisher"}, inplace=True)


freq = pd.read_csv(FREQ_FILE, sep='\s+', usecols=['SNP', 'MAF'])
df = pd.merge(log_hybrid, assoc_adjusted, on="SNP", how="right")
df = pd.merge(df, freq, on="SNP", how="left")
#merge freq with df
freq_assoc = pd.merge(assoc, df, on="SNP", how="left")

#read in recode file
recode = pd.read_csv(RECODE_FILE, sep="\s+")

# Pre-filter the dataset
cases_data = recode[recode["PHENOTYPE"] == 2]
controls_data = recode[recode["PHENOTYPE"] == 1]
# Make a list from the column names
column_names = recode.columns.tolist()

# Drop the first 6 columns to keep the variants 
variants = column_names[6:]
results = []
for variant in variants:
    # For cases
    hom_cases = cases_data[cases_data[variant] == 2].shape[0]
    het_cases = cases_data[cases_data[variant] == 1].shape[0]
    total_cases = cases_data.shape[0]
    # For controls
    hom_controls = controls_data[controls_data[variant] == 2].shape[0]
    het_controls = controls_data[controls_data[variant] == 1].shape[0]
    total_controls = controls_data.shape[0]
    results.append({
        "Variant": variant,
        "Hom Cases": hom_cases,
        "Het Cases": het_cases,
        "Total Cases": total_cases,
        "Hom Controls": hom_controls,
        "Het Controls": het_controls,
        "Total Controls": total_controls,
    })

# Return results
df_results = pd.DataFrame(results)
df_results["SNP"] = df_results["Variant"].apply(lambda x: x.rsplit("_", 1)[0])
df_results = df_results.drop("Variant", axis=1)

# Merge with assoc results 
full_results = pd.merge(freq_assoc, df_results, on="SNP", how="left")


# append variant annotation
annotations = pd.read_csv(f"{WORK_DIR}/vep_annotations_filtered_pick_extra_columns.txt", sep="\t", usecols=["Consequence", "SNP", "HGVS"])
full_results_annotated = pd.merge(full_results, annotations, on="SNP", how="left")
full_results_annotated.head()
#subset to only columns to keep

clean_full_results = full_results_annotated[["SNP", "Consequence", "HGVS", "A1", "A1_CT", "A1_FREQ", "FIRTH?",
                                             "P", "P_fisher", "BONF", "OR", "LOG(OR)_SE","L95", "U95",
                                  "F_A", "F_U", "MAF",
                                   "Hom Cases", "Het Cases", "Total Cases", 
                                   "Hom Controls","Het Controls", "Total Controls", "ERRCODE"
                                   ]].copy()



clean_full_results.insert(1, "Ancestry", ANC)

clean_full_results.head()

,SNP,Ancestry,Consequence,HGVS,A1,A1_CT,A1_FREQ,FIRTH?,P,P_fisher,...,F_A,F_U,MAF,Hom Cases,Het Cases,Total Cases,Hom Controls,Het Controls,Total Controls,ERRCODE
0,chr21:33721217:C:T,EUR,missense_variant,p.Ala23Val,T,1,0.000031,Y,0.652295,1.0000,...,0.000037,0.000000,0.000031,0,1,13413,0,0,2587,.
1,chr21:33721223:A:G,EUR,missense_variant,p.His25Arg,G,1,0.000031,Y,0.150471,0.1617,...,0.000000,0.000193,0.000031,0,0,13413,0,1,2587,.
2,chr21:33722604:CT:C,EUR,frameshift_variant,p.Gln50AsnfsTer9,C,1,0.000033,Y,0.417587,1.0000,...,0.000040,0.000000,0.000033,0,1,13413,0,0,2587,.
3,chr21:33735066:G:A,EUR,missense_variant,p.Asp70Asn,A,1,0.000031,Y,0.469445,1.0000,...,0.000037,0.000000,0.000031,0,1,13413,0,0,2587,.
4,chr21:33735084:G:A,EUR,missense_variant,p.Val76Met,A,1,0.000031,Y,0.111929,0.1617,...,0.000000,0.000193,0.000031,0,0,13413,0,1,2587,.


In [73]:
# Look at significant SNPs, if any 
sig_freq = clean_full_results[clean_full_results['P']<0.05]
sig_snps = sig_freq['SNP'].tolist()
sig_df_results = clean_full_results[clean_full_results['SNP'].isin(sig_snps)]
sig_df_results

,SNP,Ancestry,Consequence,HGVS,A1,A1_CT,A1_FREQ,FIRTH?,P,P_fisher,...,F_A,F_U,MAF,Hom Cases,Het Cases,Total Cases,Hom Controls,Het Controls,Total Controls,ERRCODE
90,chr21:33811126:C:T,EUR,missense_variant,p.Ala824Val,T,1,0.000032,Y,0.037681,0.1612,...,0.0,0.000195,0.000031,0,0,13413,0,1,2587,.


In [74]:
# save files to working directory
clean_full_results.to_csv(f'{WORK_DIR}/{ANC}_{GENE}_GP2_R{RELEASE}.fullVariantInformation.txt', sep="\t", index=False)
sig_df_results.to_csv(f'{WORK_DIR}/{ANC}_{GENE}_GP2_R{RELEASE}.SignificantVariantInformation.txt' , sep="\t", index=False)